![title](imagens/projeto20.jpg)

# Projeto 20 - Segmentação de Usuários de APP - STATSO
------------

A segmentação de usuários de aplicativos é a tarefa de agrupar usuários com base em como eles interagem com o aplicativo. 

Isso ajuda a encontrar usuários retidos, encontrar o segmento de usuários para uma campanha de marketing e resolver muitos outros problemas comerciais em que você precisa procurar usuários com características semelhantes.

Estudo de Caso disponível em STATSO (https://statso.io/app-users-segmentation-case-study/)


## Problema do Negócio a ser resolvido:

No mundo altamente competitivo dos aplicativos, empresas e desenvolvedores de aplicativos precisam entender e direcionar grupos de usuários específicos para melhorar o engajamento e reter mais usuários, aumentando o valor vitalício.

Aqui está um conjunto de dados que coletamos de um aplicativo para encontrar uma abordagem baseada em dados para segmentar os usuários do aplicativo com base em seus hábitos de uso e capacidade de gasto para encontrar usuários que o aplicativo reterá e perderá ao longo do tempo.

Abaixo estão todas as características no conjunto de tivas.
ras alternativas.


- **userid**: O número de identidade do usuário;
- **Average Screen Time**: O tempo médio de tela do usuário no aplicativo;
- **Average Spent on App (INR)**: O valor médio gasto pelo usuário no aplicativo;
- **Left Review**: O usuário deixou algum comentário sobre a experiência no aplicativo? (1 se sim, caso contrário 0);
- **Ratings**: Avaliações dadas pelo usuário ao aplicativo;
- **New Password Request**: O número de vezes que o usuário solicitou uma nova senha;
- **Last Visited Minutes**: Minutos passados desde a última atividade do usuário;
- **Status**: Instalado se o aplicativo está instalado e desinstalado se o usuário deletou o aplicativo;

Encontre relações entre os usuários que ainda estão usando o aplicativo e os usuários que desinstalaram o aplicativo e crie segmentos de usuários para entender os usuários retidos e os usuários que podem ser retidos antes de migrarem para outras alternativas.
ras alternativas.

Vamos ao trabalho!

In [ ]:
# Versão da Linguagem Python
from platform import python_version
print('Versão da Linguagem Python usada neste Projeto no Jupyter Notebook:', python_version())

In [ ]:
# Para atualizar um pacote, execute o comando abaixo no terminal ou prompt de comando:
# pip install -U nome_pacote

# Para instalar a versão exata de um pacote, execute o comando abaixo no terminal ou prompt de comando:
# !pip install nome_pacote==versão_desejada

# Depois de instalar ou atualizar o pacote, reinicie o jupyter notebook.

# Instala o pacote watermark. 
# Esse pacote é usado para gravar as versões de outros pacotes usados neste jupyter notebook.
# !pip install -q -U watermark

### Importando os pacotes / bibliotecas

In [ ]:
# Importando as bibliotecas
import pandas as pd
import numpy as np

# para os gráficos/plotagem
import plotly  #para identificação da versão
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio

# Importanto o método MinMaxScaler do Sklearn para normalização dos dados
from sklearn.preprocessing import MinMaxScaler

# Importanto o Método KMeans (algoritmo de classificação)
import sklearn  #para identificação da versão
from sklearn.cluster import KMeans

# para evitar mensagens de alerta/warnings.
import warnings
warnings.filterwarnings("ignore")

# Carregar o módulo de funções para limpeza de dados
from limpeza_dados import *

In [ ]:
# Versões dos pacotes usados neste jupyter notebook
%reload_ext watermark
%watermark -a "pyPRO - Seja um Profissional Python!" --iversions

In [ ]:
# Definindo um template
pio.templates.default = "plotly_white"

In [ ]:
# Importando o dataset
data = pd.read_csv("dados/userbehaviour.csv")
# mostrando a parte inicial do dataset
print(data.head())

Vamos começar olhando para o tempo de tela mais alto, mais baixo e médio de todos os usuários:


In [ ]:
print(f'Average Screen Time = {data["Average Screen Time"].mean()}')
print(f'Highest Screen Time = {data["Average Screen Time"].max()}')
print(f'Lowest Screen Time = {data["Average Screen Time"].min()}')

Agora vamos dar uma olhada no valor mais alto, mais baixo e na média gasta por todos os usuários:


In [ ]:
print(f'Average Spend of the Users = {data["Average Spent on App (INR)"].mean()}')
print(f'Highest Spend of the Users = {data["Average Spent on App (INR)"].max()}')
print(f'Lowest Spend of the Users = {data["Average Spent on App (INR)"].min()}')

Agora vamos dar uma olhada na relação entre a capacidade de gasto e o tempo de tela dos usuários ativos e dos usuários que desinstalaram o aplicativo:


In [ ]:
figure = px.scatter(data_frame = data, 
                    x="Average Screen Time",
                    y="Average Spent on App (INR)", 
                    size="Average Spent on App (INR)", 
                    color= "Status",
                    title = "Relationship Between Spending Capacity and Screentime",
                    trendline="ols")
figure.show()

Então, isso é ótimo! Os usuários que desinstalaram o aplicativo tinham um tempo médio de tela de menos de 5 minutos por dia, e o valor médio gasto era inferior a 100. Também podemos ver uma relação linear entre o tempo médio de tela e o gasto médio dos usuários que ainda estão usando o aplicativo.

Agora vamos dar uma olhada na relação entre as avaliações dadas pelos usuários e o tempo médio de tela:


In [ ]:
figure = px.scatter(data_frame = data, 
                    x="Average Screen Time",
                    y="Ratings", 
                    size="Ratings", 
                    color= "Status", 
                    title = "Relationship Between Ratings and Screentime",
                    trendline="ols")
figure.show()

Então, podemos ver que os usuários que desinstalaram o aplicativo deram no máximo cinco avaliações ao aplicativo. 

Seu tempo de tela é muito baixo em comparação com os usuários que deram mais avaliações. 

Portanto, isso descreve que os usuários que não gostam de passar mais tempo classificam o aplicativo como baixo e o desinstalam em algum momento.


### Segmentação de Usuários de Aplicativos para Encontrar Usuários Retidos e Perdidos
Depois de explorarmos os dados e entendermos algumas relações, chegou a hora de avançar para a segmentação de usuários de aplicativos para encontrar os usuários que o aplicativo reteve e perdeu para sempre. 

Para tanto, vamos utilizar o algoritmo de agrupamento **K-means** em Aprendizado de Máquina para esta tarefa.


In [ ]:
# Fazendo o agrupamento dos dados importantes para o modelo.
clustering_data = data[["Average Screen Time", "Left Review", 
                        "Ratings", "Last Visited Minutes", 
                        "Average Spent on App (INR)", 
                        "New Password Request"]]

In [ ]:
# Fazendo a normalização dos dados
for i in clustering_data.columns:
    MinMaxScaler(i)

In [ ]:
# realizando a classificação.
kmeans = KMeans(n_clusters=3, n_init=10)
clusters = kmeans.fit_predict(clustering_data)
data["Segments"] = clusters

In [ ]:
# apresentando os dados.
print(data.head(10))

Agora vamos dar uma olhada no número de segmentos que obtivemos:


In [ ]:
print(data["Segments"].value_counts())

In [ ]:
# Vamos renomear esse segmentos para melhor entendimento
data["Segments"] = data["Segments"].map({0: "Retained", 1: 
    "Churn", 2: "Needs Attention"})

In [ ]:
# Vamos visualizar esses segmentos graficamente
PLOT = go.Figure()
for i in list(data["Segments"].unique()):
    

    PLOT.add_trace(go.Scatter(x = data[data["Segments"]== i]['Last Visited Minutes'],
                                y = data[data["Segments"] == i]['Average Spent on App (INR)'],
                                mode = 'markers',marker_size = 6, marker_line_width = 1,
                                name = str(i)))
PLOT.update_traces(hovertemplate='Last Visited Minutes: %{x} <br>Average Spent on App (INR): %{y}')

    
PLOT.update_layout(width = 800, height = 800, autosize = True, showlegend = True,
                   yaxis_title = 'Average Spent on App (INR)',
                   xaxis_title = 'Last Visited Minutes',
                   scene = dict(xaxis=dict(title = 'Last Visited Minutes', titlefont_color = 'black'),
                                yaxis=dict(title = 'Average Spent on App (INR)', titlefont_color = 'black')))

O segmento **azul** mostra o segmento de usuários que o aplicativo reteve ao longo do tempo. 

O segmento **vermelho** indica o segmento de usuários que acabaram de desinstalar o aplicativo e em breve vamos perdê-los. 

E o segmento **verde** indica o segmento de usuários que o aplicativo perdeu.


## Resumo
Então, é assim que você pode segmentar usuários com base em como eles interagem com o aplicativo. 

A segmentação de usuários de aplicativos ajuda as empresas a encontrar usuários retidos, encontrar o segmento de usuários para uma campanha de marketing e resolver muitos outros problemas comerciais em que você precisa procurar usuários com características semelhantes.


## Fim do Projeto 20